In [1]:
import pandas as pd
import numpy as np


In [16]:
tpi = pd.read_csv('../data/Token Price Index/tpi_average_nov2024_jun2026_1.csv')
overall = pd.read_csv('../data/FRED/Overall Job Postings.csv')
software = pd.read_csv('../data/FRED/Software Job postings.csv')

tpi.rename(columns={'week_ending': 'observation_date'}, inplace=True)


In [ ]:

print(overall.head())
print(software.head())

  observation_date  USDATAWWWNGSP
0       1997-01-01          32000
1       1998-01-01          37488
2       1999-01-01          31737
3       2000-01-01          27489
4       2001-01-01          44274
  observation_date  IHLIDXUS
0       2021-06-12    132.55
1       2021-06-13    132.67
2       2021-06-14    133.00
3       2021-06-15    133.32
4       2021-06-16    133.50
  observation_date  IHLIDXUSTPSOFTDEVE
0       2021-06-12              136.92
1       2021-06-13              137.16
2       2021-06-14              137.49
3       2021-06-15              137.99
4       2021-06-16              138.26


In [17]:
target = 'observation_date'

df_merged = pd.merge(software, overall, on=target, how='outer')
df_merged = pd.merge(df_merged, tpi, on=target, how='outer')
print(df_merged.head())



  observation_date  IHLIDXUSTPSOFTDEVE  IHLIDXUS  \
0       2021-06-12              136.92    132.55   
1       2021-06-13              137.16    132.67   
2       2021-06-14              137.49    133.00   
3       2021-06-15              137.99    133.32   
4       2021-06-16              138.26    133.50   

   tpi_average_usd_per_m_tokens  member_count notes  
0                           NaN           NaN   NaN  
1                           NaN           NaN   NaN  
2                           NaN           NaN   NaN  
3                           NaN           NaN   NaN  
4                           NaN           NaN   NaN  


In [18]:
df_merged.rename(columns={'IHLIDXUS': 'Overall_postings', 'IHLIDXUSTPSOFTDEVE': 'Software_postings'}, inplace=True)

df_merged.head()

,observation_date,Software_postings,Overall_postings,tpi_average_usd_per_m_tokens,member_count,notes
0,2021-06-12,136.92,132.55,NaN,NaN,NaN
1,2021-06-13,137.16,132.67,NaN,NaN,NaN
2,2021-06-14,137.49,133.00,NaN,NaN,NaN
3,2021-06-15,137.99,133.32,NaN,NaN,NaN
4,2021-06-16,138.26,133.50,NaN,NaN,NaN


In [19]:
df_merged.to_csv('../data/combined_data.csv', index=False)

In [22]:
df = pd.read_csv("../data/combined_data.csv", parse_dates=["observation_date"])

# Compute monthly sums across all daily rows
df["_ym"] = df["observation_date"].dt.to_period("M")

monthly_sums = (
    df.groupby("_ym")[["Software_postings", "Overall_postings"]]
    .sum()
)

# Keep only rows where TPI has a value, then swap in the monthly sums
out = (
    df[df["tpi_average_usd_per_m_tokens"].notna()]
    .copy()
    .merge(monthly_sums, on="_ym", suffixes=("", "_sum"))
)

out["Software_postings"] = out["Software_postings_sum"]
out["Overall_postings"]  = out["Overall_postings_sum"]

out = out.drop(columns=["_ym", "Software_postings_sum", "Overall_postings_sum"])

out.to_csv("../data/combined_data.csv", index=False)